# Inferência e decisão A1 (lab UE-TP / load-anomaly)

Lê artefatos já gerados (`model.json`, `decision.json`) e, se o lab estiver no path, mostra como reexecutar o pipeline.

**Escopo da disciplina:** modelo simples (MAD) é suficiente. A1 **dry-run** por padrão.


In [1]:
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)
import json
import sys
import pandas as pd

SAMPLE = Path("../datasets/kpm-ue-tp-sample")
model = json.loads((SAMPLE / "model.json").read_text())
decision = json.loads((SAMPLE / "decision.json").read_text())
print("model keys:", sorted(model.keys())[:12])
print("decision:", decision.get("evaluation", {}).get("decision"))
print("apply_votes:", decision.get("evaluation", {}).get("apply_votes"))
print("anomalous:", decision.get("evaluation", {}).get("latest", {}).get("anomalous_features"))


model keys: ['algorithm', 'features', 'mad_floor', 'min_anomalous_features', 'sample_count', 'schema_version', 'score_threshold', 'trained_at', 'training_source']
decision: apply
apply_votes: 5
anomalous: ['DRB.RlcSduDelayDl', 'DRB.UEThpUl', 'RRU.PrbTotUl']


## Contexto da última amostra avaliada


In [2]:
latest = decision.get("evaluation", {}).get("latest", {})
sample = latest.get("sample", {})
scores = latest.get("scores", {})
ctx = pd.DataFrame([
    {"feature": k, "value": sample.get(k), "score": scores.get(k)}
    for k in sorted(set(sample) | set(scores))
])
display(ctx)
policy = decision.get("policy", {})
print("policy keys:", list(policy)[:10])
print(json.dumps(policy, indent=2)[:800])


,feature,value,score
0,DRB.RlcSduDelayDl,154.88,154.88
1,DRB.UEThpUl,77216.56,77212.84
2,RRU.PrbTotUl,99.00,97.00


policy keys: ['actuation', 'lab_context', 'policy_data', 'policy_id', 'policytype_id', 'ric_id', 'service_id']
{
  "actuation": {
    "mode": "emulate",
    "real": {}
  },
  "lab_context": {
    "anomalous_features": [
      "DRB.RlcSduDelayDl",
      "DRB.UEThpUl",
      "RRU.PrbTotUl"
    ],
    "decision": "apply"
  },
  "policy_data": {
    "qosObjectives": {
      "priorityLevel": 10
    },
    "scope": {
      "qosId": "qos-lab",
      "ueId": "ue-any"
    }
  },
  "policy_id": "ai-load-control-20260804204423",
  "policytype_id": "1",
  "ric_id": "ric-oran",
  "service_id": "ai-training-rapp"
}


## Reexecutar no lab (terminal)

```bash
cd ../oai-cn-gnb-nonrt-nearrt
./scripts/run_ue_tp_experiment.sh                 # offline → novos artefatos
python3 scripts/ai_policy_pipeline.py apply \
  --decision logs/experiments/ue-tp-*/decision.json   # dry-run

# Commit A1 (só com Fase 2 / PMS saudável)
# AI_POLICY_COMMIT=1 ./scripts/run_ue_tp_experiment.sh
```

Roteiro completo de demo: `docs/ROTEIRO_DEMO_E2_ANALISE_A1.md`.


In [3]:
# Estatísticas do baseline MAD (modelo real do lab)
feats = model.get("features", {})
rows = []
for name, stats in feats.items():
    rows.append({"feature": name, **stats})
display(pd.DataFrame(rows))
print("score_threshold:", model.get("score_threshold"))
print("min_anomalous_features:", model.get("min_anomalous_features"))
print("algorithm:", model.get("algorithm"))


,feature,mad,max,median,min
0,DRB.RlcSduDelayDl,0.0,218.00,0.00,0.0
1,DRB.UEThpUl,0.0,4.46,3.72,3.0
2,RRU.PrbTotUl,0.0,2.00,2.00,2.0


score_threshold: 3.5
min_anomalous_features: 2
algorithm: robust-baseline-mad
